In [1]:
library(ggplot2)
library(dplyr)
library(Seurat)
library(tidyverse)
library(tidyr)
library(stringr)

Warning message:
“package ‘dplyr’ was built under R version 4.3.2”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘Seurat’ was built under R version 4.3.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.3.3”
Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.3.2”

Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘tidyverse’ was built under R version 4.3.3”
Warning message:
“package ‘readr’ was built under R version 4.3.3”
Warning message:
“package ‘stringr’ was built under R version 4.3.2”
Warning message:
“package ‘forcats’ was built under R version 4.3.3”
── Attaching core tidyverse packages ────────────────────────────────

In [ ]:
# "indir" is a custom input path, and "outdir" is a custom output path.
# indir=""
# outdir=""

In [ ]:
specific_hyper_hypo_DHMR <- read.csv(paste0(indir,"/00-dhmr_specific_hyper_hypo_DHMR_include_region.csv"),row.names=1,check.names=F)

In [3]:
hyper_regions <- specific_hyper_hypo_DHMR %>%
  select(Subclass, Specific_Hyper_Region) %>%
  # Replace single quotes with double quotes
  mutate(Specific_Hyper_Region = str_replace_all(Specific_Hyper_Region, "'", "\"")) %>%
  # Converts a string to a real list
  mutate(Specific_Hyper_Region = map(Specific_Hyper_Region, ~ jsonlite::fromJSON(.))) %>%
  # Expand into separate rows
  unnest(Specific_Hyper_Region) %>%
  rename(region = Specific_Hyper_Region) %>% data.frame()

In [4]:
head(hyper_regions);dim(hyper_regions)

,Subclass,region
,<chr>,<chr>
1,L2/3 IT CTX Glut,chr1_40205592_40205913
2,L2/3 IT CTX Glut,chr1_63801218_63802655
3,L2/3 IT CTX Glut,chr1_95910842_95911315
4,L2/3 IT CTX Glut,chr1_121992803_121993033
5,L2/3 IT CTX Glut,chr1_123859818_123861293
6,L2/3 IT CTX Glut,chr1_127035200_127035964


[1] 122947      2

In [5]:
unique(hyper_regions$Subclass);length(unique(hyper_regions$Subclass))

[1] "L2/3 IT CTX Glut"      "L4/5 IT CTX Glut"      "L5 IT CTX Glut"       
 [4] "L2/3 IT RSP Glut"      "L4 RSP-ACA Glut"       "L5 ET CTX Glut"       
 [7] "SUB-ProS Glut"         "CA1-ProS Glut"         "CA3 Glut"             
[10] "CLA-EPd-CTX Car3 Glut" "L5 NP CTX Glut"        "L6 CT CTX Glut"       
[13] "DG Glut"               "OB Eomes Ms4a15 Glut"  "OB-in Frmd7 Gaba"     
[16] "OB-out Frmd7 Gaba"     "OB Dopa-Gaba"          "OB-STR-CTX Inh IMN"   
[19] "Sncg Gaba"             "Lamp5 Gaba"            "Pvalb Gaba"           
[22] "Sst Gaba"              "STR D1 Gaba"           "STR D2 Gaba"          
[25] "ACB-BST-FS D1 Gaba"

[1] 25

In [ ]:
process_regions <- function(subclass1,subclass2) { # 'L2/3 IT CTX Glut', 'L2_3'
  df <- hyper_regions[hyper_regions$Subclass == subclass1,]
  df_split <- do.call(rbind, strsplit(df$region, "_"))
  df_split <- as.data.frame(df_split, stringsAsFactors = FALSE)
  colnames(df_split) <- c("chr", "start", "end")
  df_split$start <- as.numeric(df_split$start)
  df_split$end <- as.numeric(df_split$end)
  df_split$mid <- (df_split$start + df_split$end) / 2
  df_split$new_start <- pmax(0, df_split$mid - 500)
  df_split$new_end <- df_split$mid + 500

  df_final <- data.frame(
    chr = df_split$chr,
    start = as.integer(df_split$new_start),
    end = as.integer(df_split$new_end),
    region = paste0(df_split$chr, ":", as.integer(df_split$new_start), "_", as.integer(df_split$new_end))
  )
  write.table(df_final,
              sprintf("%s/%s_hyper_DHMR.bed",outdir,subclass2),
              sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
  print(head(df_final))
  print(dim(df_final))
}

In [7]:
process_regions('L2/3 IT CTX Glut', 'L2_3')

   chr     start       end                   region
1 chr1  40205252  40206252   chr1:40205252_40206252
2 chr1  63801436  63802436   chr1:63801436_63802436
3 chr1  95910578  95911578   chr1:95910578_95911578
4 chr1 121992418 121993418 chr1:121992418_121993418
5 chr1 123860055 123861055 chr1:123860055_123861055
6 chr1 127035082 127036082 chr1:127035082_127036082
[1] 157   4


In [8]:
process_regions('L4/5 IT CTX Glut', 'L4_5')

   chr    start      end                 region
1 chr1  3989606  3990606   chr1:3989606_3990606
2 chr1  7536517  7537517   chr1:7536517_7537517
3 chr1  9780815  9781815   chr1:9780815_9781815
4 chr1 14839073 14840073 chr1:14839073_14840073
5 chr1 14914779 14915779 chr1:14914779_14915779
6 chr1 17756423 17757423 chr1:17756423_17757423
[1] 1392    4


In [9]:
process_regions('L5 IT CTX Glut', 'L5')

   chr    start      end                 region
1 chr1  4185672  4186672   chr1:4185672_4186672
2 chr1  9994550  9995550   chr1:9994550_9995550
3 chr1 20083023 20084023 chr1:20083023_20084023
4 chr1 22220457 22221457 chr1:22220457_22221457
5 chr1 25591789 25592789 chr1:25591789_25592789
6 chr1 26496284 26497284 chr1:26496284_26497284
[1] 632   4


In [10]:
process_regions('L5 NP CTX Glut', 'NP')

   chr   start     end               region
1 chr1 4035689 4036689 chr1:4035689_4036689
2 chr1 4933311 4934311 chr1:4933311_4934311
3 chr1 6077804 6078804 chr1:6077804_6078804
4 chr1 6367459 6368459 chr1:6367459_6368459
5 chr1 7734392 7735392 chr1:7734392_7735392
6 chr1 9367457 9368457 chr1:9367457_9368457
[1] 9693    4


In [11]:
process_regions('L6 CT CTX Glut', 'CT')

   chr    start      end                 region
1 chr1  3908393  3909393   chr1:3908393_3909393
2 chr1  5417353  5418353   chr1:5417353_5418353
3 chr1  5957907  5958907   chr1:5957907_5958907
4 chr1  9594969  9595969   chr1:9594969_9595969
5 chr1 11620744 11621744 chr1:11620744_11621744
6 chr1 11728492 11729492 chr1:11728492_11729492
[1] 1243    4


In [12]:
process_regions('DG Glut', 'DG')

   chr    start      end                 region
1 chr1  5012521  5013521   chr1:5012521_5013521
2 chr1  6862606  6863606   chr1:6862606_6863606
3 chr1  7358520  7359520   chr1:7358520_7359520
4 chr1 13910825 13911825 chr1:13910825_13911825
5 chr1 16814367 16815367 chr1:16814367_16815367
6 chr1 17029988 17030988 chr1:17029988_17030988
[1] 687   4


In [13]:
process_regions('CA1-ProS Glut', 'CA1')

   chr   start     end               region
1 chr1 3707240 3708240 chr1:3707240_3708240
2 chr1 3868330 3869330 chr1:3868330_3869330
3 chr1 4000851 4001851 chr1:4000851_4001851
4 chr1 4017553 4018553 chr1:4017553_4018553
5 chr1 4168027 4169027 chr1:4168027_4169027
6 chr1 4249638 4250638 chr1:4249638_4250638
[1] 2566    4


In [14]:
process_regions('CA3 Glut', 'CA23')

   chr   start     end               region
1 chr1 3151305 3152305 chr1:3151305_3152305
2 chr1 3776523 3777523 chr1:3776523_3777523
3 chr1 4031436 4032436 chr1:4031436_4032436
4 chr1 4057009 4058009 chr1:4057009_4058009
5 chr1 4065207 4066207 chr1:4065207_4066207
6 chr1 4072107 4073107 chr1:4072107_4073107
[1] 3974    4


In [15]:
process_regions('Pvalb Gaba', 'Pvalb')

   chr   start     end               region
1 chr1 3168678 3169678 chr1:3168678_3169678
2 chr1 3188267 3189267 chr1:3188267_3189267
3 chr1 3189154 3190154 chr1:3189154_3190154
4 chr1 3229735 3230735 chr1:3229735_3230735
5 chr1 3257828 3258828 chr1:3257828_3258828
6 chr1 3275051 3276051 chr1:3275051_3276051
[1] 13766     4


In [16]:
process_regions('Sst Gaba', 'Sst')

   chr   start     end               region
1 chr1 3074178 3075178 chr1:3074178_3075178
2 chr1 3083989 3084989 chr1:3083989_3084989
3 chr1 3091016 3092016 chr1:3091016_3092016
4 chr1 3096626 3097626 chr1:3096626_3097626
5 chr1 3098325 3099325 chr1:3098325_3099325
6 chr1 3104126 3105126 chr1:3104126_3105126
[1] 17454     4


In [17]:
process_regions('Sncg Gaba', 'Sncg')

   chr   start     end               region
1 chr1 3239626 3240626 chr1:3239626_3240626
2 chr1 3597956 3598956 chr1:3597956_3598956
3 chr1 3845937 3846937 chr1:3845937_3846937
4 chr1 7206724 7207724 chr1:7206724_7207724
5 chr1 7289662 7290662 chr1:7289662_7290662
6 chr1 7546590 7547590 chr1:7546590_7547590
[1] 5165    4


In [18]:
process_regions('Lamp5 Gaba', 'Lamp5')

   chr   start     end               region
1 chr1 3033026 3034026 chr1:3033026_3034026
2 chr1 3116806 3117806 chr1:3116806_3117806
3 chr1 3235621 3236621 chr1:3235621_3236621
4 chr1 3290843 3291843 chr1:3290843_3291843
5 chr1 3510333 3511333 chr1:3510333_3511333
6 chr1 3802417 3803417 chr1:3802417_3803417
[1] 10261     4
